In [16]:
# Cell 1: Setup
# =============
import os
import deltalake as dl

# Configuration
delta_path = "s3a://real-estate/lake/bronze/property"
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://127.0.0.1:9000")

# MinIO storage options
storage_options = {
    "aws_access_key_id": MINIO_ACCESS_KEY,
    "aws_secret_access_key": MINIO_SECRET_KEY,
    "aws_endpoint": MINIO_ENDPOINT,
    "aws_region": "us-east-1",
    "aws_s3_allow_unsafe_rename": "true"
}



In [19]:
# Cell 2: Debug Connection
# ========================
# First, let's test if we can connect to MinIO at all
import boto3
from botocore.client import Config

# Create S3 client to test connection
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Test connection
try:
    # List buckets
    buckets = s3_client.list_buckets()
    print("✅ Connected to MinIO!")
    print(f"Buckets: {[b['Name'] for b in buckets['Buckets']]}")
    
    # Check if our bucket exists
    bucket_name = "real-estate"
    if bucket_name in [b['Name'] for b in buckets['Buckets']]:
        # List objects in the Delta table path
        objects = s3_client.list_objects_v2(
            Bucket=bucket_name,
            Prefix="lake/bronze/property/_delta_log/",
            MaxKeys=5
        )
        
        if 'Contents' in objects:
            print(f"\n✅ Delta table exists at: {delta_path}")
            print(f"Delta log files found: {len(objects['Contents'])}")
            for obj in objects['Contents'][:3]:
                print(f"  - {obj['Key']}")
        else:
            print(f"\n❌ No Delta table found at: {delta_path}")
            print("The table might not exist yet. You need to run the pipeline first.")
    else:
        print(f"\n❌ Bucket '{bucket_name}' not found!")
        
except Exception as e:
    print(f"❌ Cannot connect to MinIO: {e}")
    print(f"Endpoint: {MINIO_ENDPOINT}")
    print("Make sure MinIO is running!")



✅ Connected to MinIO!
Buckets: ['real-estate']

✅ Delta table exists at: s3a://real-estate/lake/bronze/property
Delta log files found: 2
  - lake/bronze/property/_delta_log/00000000000000000000.json
  - lake/bronze/property/_delta_log/00000000000000000001.json


In [23]:
# Cell 3: Read Delta Table (if exists)
# ====================================
# Only try to read if we confirmed the table exists
if 'objects' in locals() and 'Contents' in objects:
    # Use the correct path format for delta-rs
    delta_path_s3 = delta_path.replace("s3a://", "s3://")
    
    try:
        # Try reading with version 0 first (no checkpoint needed)
        dt = dl.DeltaTable(delta_path_s3, version=0, storage_options=storage_options)
        print(f"\n📊 Delta Table Info:")
        print(f"Version: {dt.version()}")
        print(f"Files: {len(dt.file_uris())}")
        
        # Now try to get the latest version
        try:
            dt_latest = dl.DeltaTable(delta_path_s3, storage_options=storage_options)
            print(f"Latest version: {dt_latest.version()}")
        except:
            print("Using version 0 (checkpoint file not found)")
            dt = dt_latest if 'dt_latest' in locals() else dt
        
        # Load a sample
        df = dt.to_pandas()
        print(f"\nRows: {len(df)}")
        print(f"Columns: {len(df.columns)}")
        print(f"\nFirst 5 columns: {list(df.columns[:5])}")
        print(f"\nFirst few rows:")
        print(df.head(3))
        
    except Exception as e:
        print(f"\n❌ Error reading Delta table: {e}")
        
        # Alternative: Try using PyArrow directly
        print("\n🔄 Trying alternative method with PyArrow...")
        try:
            import pyarrow.parquet as pq
            import pyarrow.fs as fs
            
            # Create filesystem
            filesystem = fs.S3FileSystem(
                access_key=MINIO_ACCESS_KEY,
                secret_key=MINIO_SECRET_KEY,
                endpoint_override=MINIO_ENDPOINT.replace("http://", ""),
                scheme="http"
            )
            
            # Read parquet files directly
            parquet_path = "real-estate/lake/bronze/property"
            dataset = pq.ParquetDataset(parquet_path, filesystem=filesystem)
            df = dataset.read_pandas().to_pandas()
            
            print(f"✅ Successfully read using PyArrow!")
            print(f"Rows: {len(df)}")
            print(f"Columns: {list(df.columns[:5])}...")
            
        except Exception as e2:
            print(f"❌ PyArrow method also failed: {e2}")
else:
    print("\n⚠️  Skipping Delta table read - table doesn't exist yet")


❌ Error reading Delta table: Generic S3 error: Error after 0 retries in 5.931µs, max_retries:10, retry_timeout:180s, source:builder error for url (http://127.0.0.1:9000/real-estate/lake/bronze/property/_delta_log/_last_checkpoint)

🔄 Trying alternative method with PyArrow...
✅ Successfully read using PyArrow!
Rows: 25
Columns: ['propertyDetails_propertyId', 'propertyDetails_normalizedPrice']...


In [24]:
# Cell 4: Working Delta-rs Solution
# ==================================
# Let's try different configurations to make delta-rs work

print("🔧 Attempting different delta-rs configurations...\n")

# Configuration 1: Try without http:// in endpoint
try:
    storage_options_v1 = {
        "aws_access_key_id": MINIO_ACCESS_KEY,
        "aws_secret_access_key": MINIO_SECRET_KEY,
        "aws_endpoint": "127.0.0.1:9000",  # Without http://
        "aws_region": "us-east-1",
        "allow_http": "true",
        "aws_s3_allow_unsafe_rename": "true"
    }
    dt = dl.DeltaTable("s3://real-estate/lake/bronze/property", storage_options=storage_options_v1)
    print("✅ Config 1 worked!")
    df = dt.to_pandas()
    print(f"Rows: {len(df)}")
except Exception as e:
    print(f"❌ Config 1 failed: {str(e)[:100]}...")

# Configuration 2: Try with different URL format
try:
    storage_options_v2 = {
        "aws_access_key_id": MINIO_ACCESS_KEY,
        "aws_secret_access_key": MINIO_SECRET_KEY,
        "endpoint_url": MINIO_ENDPOINT,  # Try endpoint_url instead
        "region_name": "us-east-1",
        "aws_s3_allow_unsafe_rename": "true"
    }
    dt = dl.DeltaTable("s3://real-estate/lake/bronze/property", storage_options=storage_options_v2)
    print("\n✅ Config 2 worked!")
    df = dt.to_pandas()
    print(f"Rows: {len(df)}")
except Exception as e:
    print(f"\n❌ Config 2 failed: {str(e)[:100]}...")

# Configuration 3: Use PyArrow-based reading (most reliable for MinIO)
print("\n🎯 Using PyArrow-based approach (recommended for MinIO):")

import pyarrow.parquet as pq
import pyarrow.fs as fs
import pandas as pd

# Create filesystem
filesystem = fs.S3FileSystem(
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    endpoint_override="127.0.0.1:9000",
    scheme="http"
)

# Function to read Delta table manually
def read_delta_table_manual(path, filesystem):
    """Read Delta table by parsing the transaction log manually"""
    # Read the Delta log files
    log_path = f"{path}/_delta_log"
    
    # List all log files
    log_files = sorted([f for f in filesystem.get_file_info(fs.FileSelector(log_path)) 
                       if f.path.endswith('.json')])
    
    print(f"Found {len(log_files)} Delta log files")
    
    # For now, just read the Parquet files directly
    # In a full implementation, we'd parse the JSON logs to get the current state
    parquet_files = [f for f in filesystem.get_file_info(fs.FileSelector(path)) 
                    if f.path.endswith('.parquet')]
    
    if parquet_files:
        # Read all parquet files
        dataset = pq.ParquetDataset(
            parquet_files[0].path.rsplit('/', 1)[0], 
            filesystem=filesystem
        )
        return dataset.read_pandas().to_pandas()
    else:
        return pd.DataFrame()

# Read the data
df = read_delta_table_manual("real-estate/lake/bronze/property", filesystem)
print(f"\n✅ Successfully read Delta table using PyArrow!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns[:5])}...")
print(f"\nSample data:")
print(df.head(3))

# Save to a variable for further analysis
property_df = df

🔧 Attempting different delta-rs configurations...



thread '<unnamed>' panicked at /root/.cargo/registry/src/index.crates.io-6f17d22bba15001f/object_store-0.10.1/src/aws/credential.rs:335:43:
request valid: reqwest::Error { kind: Builder, source: RelativeUrlWithoutBase }
note: run with `RUST_BACKTRACE=1` environment variable to display a backtrace


PanicException: request valid: reqwest::Error { kind: Builder, source: RelativeUrlWithoutBase }

In [28]:
# Configuration 3: Use PyArrow-based reading (most reliable for MinIO)
print("\n🎯 Using PyArrow-based approach (recommended for MinIO):")

import pyarrow.parquet as pq
import pyarrow.fs as fs
import pandas as pd

# Create filesystem
filesystem = fs.S3FileSystem(
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    endpoint_override="127.0.0.1:9000",
    scheme="http"
)

# Function to read Delta table manually
def read_delta_table_manual(path, filesystem):
    """Read Delta table by parsing the transaction log manually"""
    # For now, just read the Parquet files directly
    # This is simpler and works well for reading
    try:
        dataset = pq.ParquetDataset(path, filesystem=filesystem)
        return dataset.read_pandas().to_pandas()
    except Exception as e:
        print(f"Error reading parquet files: {e}")
        return pd.DataFrame()

# Read the data
df = read_delta_table_manual("real-estate/lake/bronze/property", filesystem)
print(f"\n✅ Successfully read Delta table using PyArrow!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns[:5])}...")
print(f"\nSample data:")
print(df.head(3))

# Save to a variable for further analysis
property_df = df



🎯 Using PyArrow-based approach (recommended for MinIO):

✅ Successfully read Delta table using PyArrow!
Shape: (25, 2)
Columns: ['propertyDetails_propertyId', 'propertyDetails_normalizedPrice']...

Sample data:
  propertyDetails_propertyId  propertyDetails_normalizedPrice
0                 4001439152                              NaN
1                 4002289191                              NaN
2                 4002041896                              NaN


In [30]:
# Cell 5: Explore the Data
# ========================
print("\n📊 Data Exploration:")
print("=" * 50)

# Basic info
print(f"Total properties: {len(property_df)}")
print(f"Total columns: {len(property_df.columns)}")

# Data types
print("\n🔍 Column types:")
print(property_df.dtypes.value_counts())

# Check for the columns we need
important_cols = ['propertyDetails_propertyId', 'propertyDetails_normalizedPrice', 
                  'propertyDetails_cityName', 'propertyDetails_propertyType']
                  
print("\n✅ Important columns present:")
for col in important_cols:
    if col in property_df.columns:
        print(f"  - {col}: ✓")
    else:
        print(f"  - {col}: ✗ (missing)")

# Quick statistics
if 'propertyDetails_normalizedPrice' in property_df.columns:
    print(f"\n💰 Price Statistics:")
    print(f"  Min: CHF {property_df['propertyDetails_normalizedPrice'].min():,.0f}")
    print(f"  Max: CHF {property_df['propertyDetails_normalizedPrice'].max():,.0f}")
    print(f"  Mean: CHF {property_df['propertyDetails_normalizedPrice'].mean():,.0f}")
    print(f"  Median: CHF {property_df['propertyDetails_normalizedPrice'].median():,.0f}")




📊 Data Exploration:
Total properties: 25
Total columns: 2

🔍 Column types:
object     1
float64    1
Name: count, dtype: int64

✅ Important columns present:
  - propertyDetails_propertyId: ✓
  - propertyDetails_normalizedPrice: ✓
  - propertyDetails_cityName: ✗ (missing)
  - propertyDetails_propertyType: ✗ (missing)

💰 Price Statistics:
  Min: CHF nan
  Max: CHF nan
  Mean: CHF nan
  Median: CHF nan


/home/baltazarleon/anaconda3/envs/real-estate-env/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [31]:
# Cell 7: Display All Data
# ========================
print("\n📋 COMPLETE DATASET VIEW:")
print("=" * 80)

# Set pandas options to show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Display the entire dataframe
print(property_df)

# Reset display options to default
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

# Also show info about the data
print("\n📊 DataFrame Info:")
print("=" * 80)
property_df.info()


📋 COMPLETE DATASET VIEW:
   propertyDetails_propertyId  propertyDetails_normalizedPrice
0                  4001439152                              NaN
1                  4002289191                              NaN
2                  4002041896                              NaN
3                  4002330870                              NaN
4                  4001691186                              NaN
5                  4002296393                              NaN
6                  4002313503                              NaN
7                  4002369064                              NaN
8                  4002380113                              NaN
9                  4002101410                              NaN
10                 4001965619                              NaN
11                 4002238787                              NaN
12                 4002320208                              NaN
13                 4001701055                              NaN
14                 4002357211

In [34]:
# ==================================
print("\n🔍 Investigating MinIO/S3 Contents:")
print("=" * 80)

# List all files in the Delta table
bucket_name = "real-estate"
prefix = "lake/bronze/property/"

try:
    # List all objects in the property folder
    objects = s3_client.list_objects_v2(
        Bucket=bucket_name,
        Prefix=prefix,
        MaxKeys=100
    )
    
    if 'Contents' in objects:
        print(f"Found {len(objects['Contents'])} files:\n")
        
        parquet_files = []
        json_files = []
        other_files = []
        
        for obj in objects['Contents']:
            key = obj['Key']
            size = obj['Size']
            modified = obj['LastModified']
            
            if key.endswith('.parquet'):
                parquet_files.append((key, size, modified))
            elif key.endswith('.json'):
                json_files.append((key, size, modified))
            else:
                other_files.append((key, size, modified))
        
        print(f"📄 Parquet files ({len(parquet_files)}):")
        for f in parquet_files:
            print(f"  - {f[0].split('/')[-1]} ({f[1]:,} bytes, {f[2].strftime('%Y-%m-%d %H:%M')})")
            
        print(f"\n📄 JSON files ({len(json_files)}):")
        for f in json_files:
            print(f"  - {f[0].split('/')[-1]} ({f[1]:,} bytes)")
            
        # Check the content of the first JSON log file to understand the schema
        if json_files:
            print("\n📖 Reading Delta log to check schema...")
            first_log = json_files[0][0]
            
            obj = s3_client.get_object(Bucket=bucket_name, Key=first_log)
            content = obj['Body'].read().decode('utf-8')
            
            # Delta logs are newline-delimited JSON
            import json
            lines = content.strip().split('\n')
            for i, line in enumerate(lines[:2]):  # Show first 2 entries
                log_entry = json.loads(line)
                print(f"\nLog entry {i}:")
                if 'add' in log_entry:
                    print("  Type: ADD file")
                    print(f"  Path: {log_entry['add'].get('path', 'N/A')}")
                elif 'metaData' in log_entry:
                    print("  Type: METADATA")
                    schema = log_entry['metaData'].get('schemaString', {})
                    print(f"  Schema: {schema[:200]}...")  # First 200 chars
                elif 'commitInfo' in log_entry:
                    print("  Type: COMMIT")
                    print(f"  Operation: {log_entry['commitInfo'].get('operation', 'N/A')}")
    else:
        print("No files found in the Delta table location!")
        
except Exception as e:
    print(f"Error investigating MinIO: {e}")



🔍 Investigating MinIO/S3 Contents:
Found 3 files:

📄 Parquet files (1):
  - part-00001-41ad9d65-2887-4afb-812b-295c702fe9e3-c000.snappy.parquet (1,231 bytes, 2025-07-23 03:52)

📄 JSON files (2):
  - 00000000000000000000.json (1,308 bytes)
  - 00000000000000000001.json (1,183 bytes)

📖 Reading Delta log to check schema...

Log entry 0:

Log entry 1:
  Type: METADATA
  Schema: {"type":"struct","fields":[{"name":"propertyDetails_propertyId","type":"string","nullable":true,"metadata":{}},{"name":"propertyDetails_normalizedPrice","type":"long","nullable":true,"metadata":{}}]}...


In [35]:
# Cell 10: Analyze What We Have
# ==============================
print("\n🔍 Analyzing Current Scraper Results:")
print("=" * 80)

# Show what we actually got
print(f"Total properties scraped: {len(property_df)}")
print(f"Properties with prices: {property_df['propertyDetails_normalizedPrice'].notna().sum()}")
print(f"Properties without prices: {property_df['propertyDetails_normalizedPrice'].isna().sum()}")

# Show all property IDs we have
print("\n🏠 Property IDs scraped:")
for idx, prop_id in enumerate(property_df['propertyDetails_propertyId']):
    price = property_df.loc[idx, 'propertyDetails_normalizedPrice']
    price_str = f"CHF {price:,.0f}" if pd.notna(price) else "No price"
    print(f"  {idx+1}. ID: {prop_id} - {price_str}")

# Construct ImmoScout24 URLs to check manually
print("\n🔗 URLs to check these properties:")
print("(You can click these to see what the scraper found)")
for prop_id in property_df['propertyDetails_propertyId'].head(5):
    print(f"  https://www.immoscout24.ch/en/d/{prop_id}")


🔍 Analyzing Current Scraper Results:
Total properties scraped: 25
Properties with prices: 0
Properties without prices: 25

🏠 Property IDs scraped:
  1. ID: 4001439152 - No price
  2. ID: 4002289191 - No price
  3. ID: 4002041896 - No price
  4. ID: 4002330870 - No price
  5. ID: 4001691186 - No price
  6. ID: 4002296393 - No price
  7. ID: 4002313503 - No price
  8. ID: 4002369064 - No price
  9. ID: 4002380113 - No price
  10. ID: 4002101410 - No price
  11. ID: 4001965619 - No price
  12. ID: 4002238787 - No price
  13. ID: 4002320208 - No price
  14. ID: 4001701055 - No price
  15. ID: 4002357211 - No price
  16. ID: 4001213536 - No price
  17. ID: 4002346221 - No price
  18. ID: 4002130045 - No price
  19. ID: 4002384556 - No price
  20. ID: 4002178545 - No price
  21. ID: 4002047801 - No price
  22. ID: 4002121909 - No price
  23. ID: 4001959857 - No price
  24. ID: 4002032660 - No price
  25. ID: 4002307073 - No price

🔗 URLs to check these properties:
(You can click these to se

In [42]:
import requests
from bs4 import BeautifulSoup
import re

url = 'https://www.immoscout24.ch/en/house/buy/city-bern?pn=1&r=7&se=16&map=1'

ids = []
html = requests.get(url)
soup = BeautifulSoup(html.text, "html.parser")
links = soup.find_all('a', href=True)
hrefs = [item['href'] for item in links]
hrefs_filtered = [href for href in hrefs if href.startswith('/en/d')]
ids += [re.find_all('\d+', item)[0] for item in hrefs_filtered]

print(ids)


[]
